# Stage 7: Tuning tree models

This notebook tunes the decision tree and random forest models owned by Dev A. Search is performed on the training split only, using average precision as the selection metric.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    repo_root = Path("/Users/leshakamadara/bank-LM-prediction")
sys.path.insert(0, str(repo_root))

from src.data import get_split
from src.evaluate import cv_report, save_result
from src.pipeline import build

X_train, X_test, y_train, y_test = get_split()
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
report_path = repo_root / "reports" / "model_comparison.csv"
best_params_path = repo_root / "reports" / "best_params.json"
print("Training shape:", X_train.shape)

Training shape: (36168, 15)


## Grid search versus random search

GridSearchCV tries every combination in a small, deliberate grid, so it is useful when the decision-tree search space is manageable. RandomizedSearchCV samples a fixed number of combinations, so it explores a wider random-forest space with a predictable computation budget. Both searches use the same five stratified folds and optimize average precision.

## 1. Decision-tree grid search

In [2]:
tree_pipeline = build(DecisionTreeClassifier(class_weight="balanced", random_state=42))
tree_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 4, 5, 6, 8, 10, None],
    "model__min_samples_leaf": [10, 25, 50, 100],
}

tree_search = GridSearchCV(
    tree_pipeline,
    param_grid=tree_grid,
    scoring="average_precision",
    cv=folds,
    n_jobs=-1,
    return_train_score=True,
)
tree_search.fit(X_train, y_train)
print("Best decision-tree parameters:", tree_search.best_params_)
print(f"Best CV PR-AUC: {tree_search.best_score_:.4f}")

Best decision-tree parameters: {'model__criterion': 'gini', 'model__max_depth': None, 'model__min_samples_leaf': 50}
Best CV PR-AUC: 0.4029


In [3]:
tree_best_result = cv_report("Decision tree-tuned", tree_search.best_estimator_, X_train, y_train)
save_result(tree_best_result, report_path)

tree_top10 = pd.DataFrame(tree_search.cv_results_).sort_values("rank_test_score").head(10)
tree_top10[["rank_test_score", "mean_test_score", "std_test_score", "param_model__criterion", "param_model__max_depth", "param_model__min_samples_leaf"]]

,rank_test_score,mean_test_score,std_test_score,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf
26,1,0.402857,0.013289,gini,None,50
27,2,0.401754,0.010182,gini,None,100
22,3,0.397772,0.008278,gini,10,50
55,4,0.397540,0.009597,entropy,None,100
54,5,0.396557,0.012940,entropy,None,50
21,6,0.394476,0.013989,gini,10,25
23,7,0.394133,0.010866,gini,10,100
50,8,0.393836,0.008096,entropy,10,50
49,9,0.393752,0.014153,entropy,10,25
51,10,0.391950,0.008339,entropy,10,100


## Decision and reason

The search uses the `model__` prefix because the classifier is inside the shared pipeline. The top rows show whether depth, leaf size, or the split criterion is most common among strong candidates.

## 2. Random-forest randomized search

In [4]:
forest_pipeline = build(
    RandomForestClassifier(
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )
)
forest_distributions = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 6, 10, 15, 20],
    "model__min_samples_leaf": [1, 5, 10, 25, 50],
    "model__max_features": ["sqrt", "log2", None, 0.5],
}

forest_search = RandomizedSearchCV(
    forest_pipeline,
    param_distributions=forest_distributions,
    n_iter=25,
    scoring="average_precision",
    cv=folds,
    n_jobs=-1,
    random_state=42,
    return_train_score=True,
)
forest_search.fit(X_train, y_train)
print("Best random-forest parameters:", forest_search.best_params_)
print(f"Best CV PR-AUC: {forest_search.best_score_:.4f}")

Best random-forest parameters: {'model__n_estimators': 500, 'model__min_samples_leaf': 10, 'model__max_features': 0.5, 'model__max_depth': 20}
Best CV PR-AUC: 0.4375


In [5]:
forest_cv_results = pd.DataFrame(forest_search.cv_results_)
forest_best_row = forest_cv_results.loc[forest_search.best_index_]
forest_best_result = {
    "name": "Random forest-tuned",
    "pr_auc_mean": forest_best_row["mean_test_score"],
    "pr_auc_std": forest_best_row["std_test_score"],
    "train_pr_auc_mean": forest_best_row["mean_train_score"],
    "train_pr_auc_std": forest_best_row["std_train_score"],
    "roc_auc_mean": float("nan"),
    "roc_auc_std": float("nan"),
    "f1_mean": float("nan"),
    "f1_std": float("nan"),
    "precision_mean": float("nan"),
    "precision_std": float("nan"),
    "recall_mean": float("nan"),
    "recall_std": float("nan"),
    "fit_time_mean": forest_best_row["mean_fit_time"],
}
save_result(forest_best_result, report_path)

forest_top10 = forest_cv_results.sort_values("rank_test_score").head(10)
forest_top10[["rank_test_score", "mean_test_score", "std_test_score", "param_model__n_estimators", "param_model__max_depth", "param_model__min_samples_leaf", "param_model__max_features"]]

,rank_test_score,mean_test_score,std_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features
22,1,0.437522,0.013917,500,20,10,0.5
19,2,0.435176,0.016163,200,20,5,0.5
13,3,0.433585,0.009700,100,None,25,0.5
10,4,0.433452,0.011913,200,None,10,sqrt
3,5,0.433341,0.010878,300,10,10,None
0,6,0.433338,0.010353,200,10,10,None
6,7,0.433116,0.011834,200,20,10,sqrt
18,8,0.432967,0.010673,500,10,10,0.5
8,9,0.429473,0.012391,300,15,5,log2
17,10,0.427198,0.009727,100,20,50,None


## Decision and reason

The randomized search already measured the selected forest with five-fold average precision, so the result row reuses that recorded CV score instead of fitting another five expensive forests. The top rows show whether deeper trees, larger leaves, more estimators, or the feature-sampling choice matters most for average precision.

## 3. Preserve best parameters

In [6]:
if best_params_path.exists() and best_params_path.stat().st_size > 0:
    best_params = json.loads(best_params_path.read_text())
else:
    best_params = {}

best_params["DecisionTree-tuned"] = tree_search.best_params_
best_params["RandomForest-tuned"] = forest_search.best_params_
best_params_path.parent.mkdir(parents=True, exist_ok=True)
best_params_path.write_text(json.dumps(best_params, indent=2) + "\n")
display(pd.DataFrame(best_params).T)

,model__criterion,model__max_depth,model__min_samples_leaf,model__n_estimators,model__max_features,model__max_leaf_nodes,model__learning_rate,model__l2_regularization,model__n_neighbors,model__p,model__weights
DecisionTree-tuned,gini,None,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RandomForest-tuned,NaN,20.0,10.0,500.0,0.5,NaN,NaN,NaN,NaN,NaN,NaN
HistGradientBoosting-tuned,NaN,NaN,50.0,NaN,NaN,31.0,0.1,10.0,NaN,NaN,NaN
KNeighbors-tuned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51,1,distance


## 4. Baseline versus tuned models

In [7]:
if report_path.exists() and report_path.stat().st_size > 0:
    results = pd.read_csv(report_path)
else:
    results = pd.DataFrame()

baseline_names = ["Decision tree depth 6 leaf 50", "Random forest"]
comparison_names = baseline_names + ["Decision tree-tuned", "Random forest-tuned"]
baseline_vs_tuned = results[results["name"].isin(comparison_names)].copy()
baseline_vs_tuned[["name", "pr_auc_mean", "pr_auc_std", "train_pr_auc_mean", "roc_auc_mean", "f1_mean"]].sort_values("pr_auc_mean", ascending=False)

,name,pr_auc_mean,pr_auc_std,train_pr_auc_mean,roc_auc_mean,f1_mean
10,Random forest-tuned,0.437522,0.013917,0.700831,NaN,NaN
5,Random forest,0.411338,0.015292,1.000000,0.776984,0.424500
9,Decision tree-tuned,0.402857,0.013289,0.490113,0.756731,0.360107
4,Decision tree depth 6 leaf 50,0.347166,0.007796,0.365758,0.749013,0.392258


## Decision and reason

The tuned rows use names ending in `-tuned`, so they can be compared directly with the baseline rows without deleting earlier results. A useful tuning change should improve CV PR-AUC while keeping the train-to-CV gap reasonable.